# Phase 2 — Chunking Experiments

Goal: run all 5 chunking strategies (`fixed`, `recursive`, `sentence_window`, `parent_child`, `semantic`) against a real parsed filing (AAPL 10-K 2023, produced by Phase 1's `scripts/parse_filings.py`) and compare chunk count / size distribution, to justify the default strategy set in `configs/base.yaml` (currently `recursive`).

Also demos `TextCleaner` and `TableSerializer` output on real extracted text/tables.

See `docs/phases/phase2_processing.md` for the full design writeup.


In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from omegaconf import OmegaConf

from src.ingestion.models import ParsedFiling
from src.processing.chunkers import chunk_text
from src.processing.cleaner import TextCleaner
from src.processing.table_serializer import TableSerializer
from src.utils.config import load_config

cfg = load_config()
filing = ParsedFiling.model_validate_json(
    Path("../data/processed/AAPL/10-K_2023/parsed.json").read_text()
)
print(f"{filing.ticker} {filing.form} FY{filing.fiscal_year} — {filing.company_name}")
print(
    f"{len(filing.sections)} sections, {len(filing.raw_tables)} raw tables, "
    f"{len(filing.full_text):,} chars full_text"
)

AAPL 10-K FY2023 — Apple Inc.
23 sections, 66 raw tables, 216,621 chars full_text


## 1. Pick one section to experiment on

Use the longest Item section (richest text, most representative of real chunking load) rather than the whole filing — keeps the strategy comparisons below fast and readable.


In [2]:
section = max(filing.sections, key=lambda s: len(s.text))
print(f"{section.item_label} — {section.title!r}")
print(f"{len(section.text):,} chars")
print(section.text[:500])

Item 1A — 'Risk Factors The Company’s business, reputation, results of'
67,875 chars
Item 1A. Risk Factors The Company’s business, reputation, results of operations, financial condition and stock price can be affected by a number of factors, whether currently known or unknown, including those described below. When any one or more of these risks materialize from time to time, the Company’s business, reputation, results of operations, financial condition and stock price can be materially and adversely affected. Because of the following factors, as well as other factors affecting t


## 2. TextCleaner

`html_parser.py` already collapses whitespace and strips the ToC block. `TextCleaner` handles what's left: Unicode quote/dash normalization and stray "Table of Contents" cross-references. Real AAPL text rarely has these, so demo against a synthetic snippet with the artifacts injected.


In [3]:
cleaner = TextCleaner()

dirty = (
    section.text[:200]
    + " See Table of Contents above for more. The Company's revenue – net of returns — grew."
)
clean = cleaner.clean(dirty)
print("BEFORE:", dirty[-90:])
print("AFTER: ", clean[-70:])

BEFORE: cludi See Table of Contents above for more. The Company's revenue – net of returns — grew.
AFTER:  udi See above for more. The Company's revenue - net of returns - grew.


## 3. TableSerializer on a real extracted table

AAPL 10-K financial statement tables — check markdown output keeps row/column structure, and NL sentences pull out (label, period, value) triples for dense-embedding recall.


In [4]:
serializer = TableSerializer()
serialized = serializer.serialize_all(filing.raw_tables)
print(f"{len(filing.raw_tables)} raw tables -> {len(serialized)} serialized (non-empty)")
print()
print(serialized[0] if serialized else "(no tables parsed)")

66 raw tables -> 57 serialized (non-empty)

| California |  | 94-2404110 |
| --- | --- | --- |
| (State or other jurisdiction of incorporation or organization) |  | (I.R.S. Employer Identification No.) |
| One Apple Park Way |  |  |
| Cupertino , California |  | 95014 |
| (Address of principal executive offices) |  | (Zip Code) |

(State or other jurisdiction of incorporation or organization) (94-2404110): (I.R.S. Employer Identification No.). Cupertino , California (94-2404110): 95014. (Address of principal executive offices) (94-2404110): (Zip Code)


## 4. Compare chunking strategies on the same section text

Run `fixed`, `recursive`, `sentence_window`, `parent_child` against the cleaned section text using the same base config, only overriding `chunking.strategy` each time (`semantic` is in its own cell below — it lazy-loads BGE-M3, ~2.3GB).


In [5]:
cleaned_text = cleaner.clean(section.text)

strategies = ["fixed", "recursive", "sentence_window", "parent_child"]
results = {}

for strategy in strategies:
    cfg_variant = OmegaConf.merge(cfg, {"chunking": {"strategy": strategy}})
    chunks = chunk_text(cleaned_text, cfg_variant)
    indexed = [
        c for c in chunks if not c.is_parent
    ]  # exclude parent-only context chunks from size stats
    lengths = [len(c.text) for c in indexed]
    results[strategy] = {
        "n_chunks": len(chunks),
        "n_indexed": len(indexed),
        "avg_len": round(sum(lengths) / len(lengths), 1) if lengths else 0,
        "min_len": min(lengths) if lengths else 0,
        "max_len": max(lengths) if lengths else 0,
    }

for strategy, stats in results.items():
    print(f"{strategy:16s} {stats}")

fixed            {'n_chunks': 152, 'n_indexed': 152, 'avg_len': 510.1, 'min_len': 227, 'max_len': 512}
recursive        {'n_chunks': 174, 'n_indexed': 174, 'avg_len': 394.3, 'min_len': 81, 'max_len': 511}
sentence_window  {'n_chunks': 313, 'n_indexed': 313, 'avg_len': 215.9, 'min_len': 8, 'max_len': 1065}
parent_child     {'n_chunks': 200, 'n_indexed': 166, 'avg_len': 455.5, 'min_len': 227, 'max_len': 512}


## 5. Semantic chunking (optional — downloads BGE-M3, ~2.3GB on first run)

Set `RUN_SEMANTIC = True` to actually run this. Skipped by default so re-running this notebook stays fast on a machine without the model cached.


In [6]:
RUN_SEMANTIC = False

if RUN_SEMANTIC:
    cfg_semantic = OmegaConf.merge(cfg, {"chunking": {"strategy": "semantic"}})
    semantic_chunks = chunk_text(cleaned_text, cfg_semantic)
    lengths = [len(c.text) for c in semantic_chunks]
    print(f"semantic: {len(semantic_chunks)} chunks, avg_len={sum(lengths) / len(lengths):.1f}")
    for c in semantic_chunks[:3]:
        print("---")
        print(c.text[:200])
else:
    print("skipped — set RUN_SEMANTIC = True to run")

skipped — set RUN_SEMANTIC = True to run


## 6. Full filing, end-to-end (matches `scripts/process_filings.py`)

Sanity-check the actual on-disk output produced by the CLI script (`python scripts/process_filings.py --ticker AAPL --years 2023`), which uses the default config strategy (`recursive`).


In [7]:
import json

chunks_on_disk = json.loads(Path("../data/processed/AAPL/10-K_2023/chunks.json").read_text())
print(f"{len(chunks_on_disk)} chunks written to disk (strategy={chunks_on_disk[0]['strategy']!r})")
print()
print(chunks_on_disk[0]["text"][:300])
print(chunks_on_disk[0]["metadata"])

737 chunks written to disk (strategy='recursive')

Item 1. Business Company Background The Company designs, manufactures and markets smartphones, personal computers, tablets, wearables and accessories, and sells a variety of related services. The Company's fiscal year is the 52- or 53-week period that ends on the last Saturday of September. Products
{'ticker': 'AAPL', 'cik': '0000320193', 'form': '10-K', 'fiscal_year': 2023, 'company_name': 'Apple Inc.', 'item_label': 'Item 1', 'section_title': 'Business Company Background The Company designs, manufactures and'}


## Conclusion

- `recursive` (current `configs/base.yaml` default) gives well-bounded chunk sizes without the mid-sentence cuts `fixed` produces — kept as default.
- `sentence_window` and `parent_child` trade index size for retrieval/generation context tradeoffs (see `docs/phases/phase2_processing.md`) — worth A/B testing against `recursive` once Phase 3 (Indexing) + Phase 7 (Evaluation/RAGAS) exist, logged in the Experiment Log table in `CLAUDE.md`.
- `semantic` chunking is implemented but not benchmarked here by default (model download cost) — run cell 5 with `RUN_SEMANTIC = True` once BGE-M3 is cached locally.
- Real run via `scripts/process_filings.py` on AAPL 10-K 2023 produced 737 chunks with the default `recursive` strategy — output written to `data/processed/AAPL/10-K_2023/chunks.json`, ready for Phase 3.
